In [1]:
import pandas as pd
import numpy as np


In [2]:
df=pd.read_csv('../data/interim/listings_parsed_v7.csv', dtype={"jnc": str})    


In [3]:

intervals=pd.read_csv('../data/interim/quantile_intervals_groupkfold.csv', index_col=0)

comps=pd.read_csv('../data/interim/nn_comps.csv', index_col=0)

In [4]:
def build_valuation_table(df, intervals, comps):
    base_cols = [
    "jnc", "Price_yen", "municipality", "area_m2", "age_years",
    "walk_min_nearest", "n_rooms", "has_L", "has_D", "has_K", "has_S", "is_1R",
    "building_key", "lookalike_id"
]

    interval_cols = ["lo2", "hi2", "mid"]
    if not set(interval_cols).issubset(intervals.columns):
        interval_cols = ["lo", "hi", "mid"]

    val = (
        df[base_cols]
        .join(intervals[interval_cols])
        .join(comps[["comp_est", "comp_lo", "comp_hi", "comp_n"]])
        .rename(columns={
            interval_cols[0]: "lo",
            interval_cols[1]: "hi",
            "mid": "estimate",
        })
    )

    # Convert log-space predictions to yen if necessary.
    if val[["lo", "estimate", "hi"]].median().median() < 30:
        val[["lo", "estimate", "hi"]] = np.expm1(
            val[["lo", "estimate", "hi"]]
        )

    assert len(val) == len(df)
    assert val[["estimate", "lo", "hi"]].notna().all().all()
    assert (val["lo"] <= val["estimate"]).all()
    assert (val["estimate"] <= val["hi"]).all()

    val["position"] = np.select(
        [
            val["Price_yen"] > val["hi"],
            val["Price_yen"] < val["lo"],
        ],
        ["above", "below"],
        default="within",
    )

    val["gap_pct"] = (val["Price_yen"] - val["estimate"]) / val["estimate"]

    val["comp_position"] = np.select(
        [
            val["Price_yen"] > val["comp_hi"],
            val["Price_yen"] < val["comp_lo"],
        ],
        ["above", "below"],
        default="within",
    )

    val["region"] = np.select(
        [
            val["municipality"].astype(str).str.endswith("区"),
            val["municipality"].astype(str).str.endswith("市"),
        ],
        ["ward", "city"],
        default="town_village",
    )

    val["confidence"] = np.where(
        (val["region"] == "town_village")
        | (val["area_m2"] > 80)
        | (val["comp_n"] < 5),
        "low",
        "normal",
    )

    val["label"] = np.select(
        [
            (val["position"] == "above") & (val["comp_position"] == "above"),
            (val["position"] == "above"),
            (val["position"] == "below") & (val["comp_position"] == "below"),
            (val["position"] == "below"),
        ],
        [
            "Relatively high vs model and similar listings",
            "Above the model's expected range",
            "Relatively low vs model and similar listings",
            "Below the model's expected range",
        ],
        default="Within the expected range",
    )

    return val


val = build_valuation_table(df, intervals, comps)

print("Rows:", len(val))
print("\nNaN counts:")
print(val.isna().sum())

print("\nPosition shares (%):")
print(val["position"].value_counts(normalize=True).mul(100).round(2))

print("\nLow-confidence share (%):")
print((val["confidence"].eq("low").mean() * 100).round(2))

sample_cols = (
    val.select_dtypes(exclude=["object", "string", "category"])
    .columns.difference(["jnc", "building_key"], sort=False)
)

print("\nRandom sample:")
display(val.sample(5, random_state=42)[sample_cols])


assert val["comp_n"].max() <= 10
assert (val["comp_est"] != val["estimate"]).mean() > 0.99
assert 0.78 < (val["position"] == "within").mean() < 0.82

val.to_csv("../data/interim/valuation_table.csv", index=True)

Rows: 183490

NaN counts:
jnc                    0
Price_yen              0
municipality           0
area_m2                0
age_years              0
walk_min_nearest    1665
n_rooms                0
has_L                  0
has_D                  0
has_K                  0
has_S                  0
is_1R                  0
building_key           0
lookalike_id           0
lo                     0
hi                     0
estimate               0
comp_est               0
comp_lo                0
comp_hi                0
comp_n                 0
position               0
gap_pct                0
comp_position          0
region                 0
confidence             0
label                  0
dtype: int64

Position shares (%):
position
within    79.89
above     10.14
below      9.97
Name: proportion, dtype: float64

Low-confidence share (%):
2.79

Random sample:


,Price_yen,area_m2,age_years,walk_min_nearest,n_rooms,has_L,has_D,has_K,has_S,is_1R,lookalike_id,lo,hi,estimate,comp_est,comp_lo,comp_hi,comp_n,gap_pct
59113,97000,46.31,14,7.0,1,1,1,1,0,0,59384,73106.625000,126635.843750,93406.054688,104143.036512,101847.241512,110724.469633,10.0,0.038477
119992,126000,25.84,1,9.0,1,0,0,1,0,0,37395,102299.445312,135031.609375,122719.617188,137772.511848,137772.511848,138793.048973,10.0,0.026731
104736,90000,31.88,24,3.0,1,0,1,1,0,0,44566,77092.867188,116859.304688,92193.515625,91074.806516,83654.814351,117016.067167,10.0,-0.023793
94546,69000,21.12,5,8.0,1,0,0,0,0,1,144623,73481.914062,97593.921875,82859.601562,97959.183673,83355.206847,100897.959184,10.0,-0.167266
76622,70000,35.00,46,5.0,1,0,1,1,0,0,152487,68728.695312,118102.484375,88143.187500,87943.192734,84045.524377,94424.235163,10.0,-0.205838


In [5]:
print(val["comp_n"].describe())       # real NN output: max <= 10
print((val["comp_est"] == val["estimate"]).mean())   # should be near 0, not 1
print((val["comp_n"] < 5).sum())

count    183490.000000
mean          9.999869
std           0.032348
min           2.000000
25%          10.000000
50%          10.000000
75%          10.000000
max          10.000000
Name: comp_n, dtype: float64
0.0
3


In [6]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
if not (project_root / "src").is_dir():
    raise FileNotFoundError(f"Could not find src directory: {project_root / 'src'}")

sys.path.insert(0, str(project_root))

from src.recommender import recommend, madori_label

val = pd.read_csv("../data/interim/valuation_table.csv", index_col=0)

rooms = pd.read_csv(
    "../data/interim/listings_parsed_v7.csv",
    usecols=["Rooms"],
)["Rooms"]

labels = val.apply(madori_label, axis=1)
print((labels == rooms.values).mean())

out, msg = recommend(
    val,
    {"max_rent": 100000, "madori": ["1LDK", "2LDK"], "max_walk": 10},
)

print(len(out), msg)
display(out.head())

1.0
20 Dropped 18 low-confidence row(s). Excluded 143 listing(s) with a gap beyond 30% (likely model error). Dropped 1277 duplicate look-alike row(s). Ranked by relatively low asking rent vs similar listings, walk, age and area.
Not a guarantee of a good deal.


,Price_yen,estimate,lo,hi,position,gap_pct,walk_min_nearest,age_years,area_m2,municipality,n_rooms,score,units_in_building
100442,80000,103323.273438,78455.226562,140699.281250,within,-0.225731,1.0,41,43.27,墨田区,1,1.0,1
29323,72000,87159.906250,76944.054688,178871.921875,below,-0.173932,1.0,21,51.30,福生市,2,1.0,2
145253,86000,95154.375000,77347.601562,131921.312500,within,-0.096206,1.0,17,42.50,立川市,1,1.0,1
53677,72500,94319.375000,73456.132812,121771.101562,below,-0.231335,1.0,19,51.67,小平市,2,1.0,1
31759,97000,106779.367188,82510.187500,143230.187500,within,-0.091585,1.0,28,48.34,北区,1,1.0,1


In [7]:
print(out[["Price_yen","estimate","gap_pct","walk_min_nearest","age_years","area_m2"]].head(10))
print((out["gap_pct"] <= -0.29).sum())

        Price_yen       estimate   gap_pct  walk_min_nearest  age_years  \
100442      80000  103323.273438 -0.225731               1.0         41   
29323       72000   87159.906250 -0.173932               1.0         21   
145253      86000   95154.375000 -0.096206               1.0         17   
53677       72500   94319.375000 -0.231335               1.0         19   
31759       97000  106779.367188 -0.091585               1.0         28   
98443      100000   96879.414062  0.032211               1.0         36   
155206     100000  100708.812500 -0.007038               1.0          4   
155181      97000   90224.000000  0.075102               1.0         29   
154767     100000  114121.765625 -0.123743               1.0         28   
95665       83000   95248.617188 -0.128596               1.0         31   

        area_m2  
100442    43.27  
29323     51.30  
145253    42.50  
53677     51.67  
31759     48.34  
98443     49.44  
155206    33.12  
155181    47.04  
154767    51

In [8]:
import src.recommender, inspect
print(src.recommender.__file__)
print("units_in_building" in inspect.getsource(src.recommender.recommend))

c:\Users\priya\Desktop\data science try\tokyo-rental-intelligence\src\recommender.py
True


In [9]:
from src.stats import municipality_stats
df = pd.read_csv("../data/interim/listings_parsed_v7.csv", dtype={"jnc": str},
                 usecols=["municipality","Price_yen","area_m2","age_years"])
s = municipality_stats(df)
assert len(s) == 52 and s["n"].sum() == 183490
print(s.head(5)); print(s.tail(5))
print(s.sort_values("median_rent_per_m2", ascending=False).head(3)["municipality"].tolist())
s.to_csv("../data/interim/municipality_stats.csv", index=False)

  municipality     n  median_rent  p25_rent  p75_rent  median_area  \
0          中央区  3858     153000.0  107250.0  200000.0        39.95   
1           港区  5005     150000.0  120000.0  250000.0        30.84   
2         千代田区  1455     138000.0  111000.0  188000.0        29.33   
3          渋谷区  4544     133000.0   95000.0  200000.0        30.53   
4          新宿区  6254     119500.0   87000.0  147000.0        25.80   

   median_rent_per_m2  median_age region  low_sample  
0              4348.6        13.0   ward       False  
1              4872.0        14.0   ward       False  
2              4597.7        13.0   ward       False  
3              4500.5        14.0   ward       False  
4              4302.2        11.0   ward       False  
   municipality     n  median_rent  p25_rent  p75_rent  median_area  \
47          羽村市   797      56000.0   43000.0   68000.0        40.92   
48          青梅市  1255      54000.0   42000.0   64000.0        43.08   
49          多摩市  2572      52000.0  

In [10]:
s.sort_values("median_rent_per_m2").head(3)

,municipality,n,median_rent,p25_rent,p75_rent,median_area,median_rent_per_m2,median_age,region,low_sample
51,奥多摩町,3,49000.0,44500.0,89500.0,40.43,1163.1,28.0,town_village,True
42,あきる野市,689,59000.0,49000.0,67000.0,50.01,1223.5,23.0,city,False
39,日の出町,82,61000.0,57000.0,70000.0,48.23,1247.1,10.0,town_village,True


In [13]:
from src.predict import valuate

feature_cols = [
    "municipality", "area_m2", "age_years", "walk_min_nearest",
    "n_rooms", "has_L", "has_D", "has_K", "has_S", "is_1R",
]

attrs = val.iloc[0][feature_cols].to_dict()

for rent in (40000, 62700, 70000, 90000):
    r = valuate(attrs, asking_rent=rent)
    print(
        rent,
        r["lo"],
        r["estimate"],
        r["hi"],
        r["position"],
        round(r["gap_pct"], 3),
    )

40000 144052.0 164464.0 188036.0 below -0.757
62700 144052.0 164464.0 188036.0 below -0.619
70000 144052.0 164464.0 188036.0 below -0.574
90000 144052.0 164464.0 188036.0 below -0.453
